# 1) Create source sound collection

Builds a collection of sounds by running a set of Freesound queries and concatenating the
results. Metadata for each sound is saved to a CSV, and an OGG preview of each is downloaded.

Set `COLLECTION` below to pick which collection to build. Each collection writes to its own
CSV and its own audio directory, so building one does not overwrite another.

Requires a Freesound API key in a `.env` file at the repo root; copy `.env.template` to `.env`
and fill it in. Get a key at https://freesound.org/apiv2/apply/.

Sound metadata comes from the `freesound` package (https://github.com/mtg/freesound-python);
see the [API documentation](http://freesound.org/docs/api/) for query syntax.

In [ ]:
import os

import freesound
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(os.path.join('..', '.env'))

FREESOUND_API_KEY = os.environ['FREESOUND_API_KEY']

FREESOUND_STORE_METADATA_FIELDS = ['id', 'name', 'username', 'previews', 'license', 'tags']

freesound_client = freesound.FreesoundClient()
freesound_client.set_token(FREESOUND_API_KEY)

In [ ]:
# The two collections compared in the paper. Durations are capped per query because
# shorter recordings are more likely to be single-shot sounds rather than field
# recordings; cows and sheep are given longer because they take longer to vocalise.
COLLECTIONS = {
    'barnyard': [
        {'num_results': 10, 'query': 'cat meow', 'filter': 'duration:[0 TO 5]', 'sort': 'rating_desc'},
        {'num_results': 20, 'query': 'dog bark', 'filter': 'duration:[0 TO 1]', 'sort': 'rating_desc'},
        {'num_results': 20, 'query': 'cow moo', 'filter': 'duration:[0 TO 10]', 'sort': 'rating_desc'},
        {'num_results': 5, 'query': 'horse whinny', 'filter': 'duration:[0 TO 5]', 'sort': 'rating_desc'},
        {'num_results': 5, 'query': 'horse neighing', 'filter': 'duration:[0 TO 5]', 'sort': 'rating_desc'},
        {'num_results': 20, 'query': 'bird chirp', 'filter': 'duration:[0 TO 10]', 'sort': 'rating_desc'},
        {'num_results': 20, 'query': 'bleat', 'filter': 'duration:[0 TO 10]', 'sort': 'rating_desc'},
    ],
    'violin': [
        {'num_results': 100, 'query': 'violin', 'filter': 'ac_single_event:True', 'sort': None},
    ],
}

COLLECTION = 'barnyard'

FILES_DIR = 'files_{0}'.format(COLLECTION)
DATAFRAME_FILENAME = 'dataframe_{0}.csv'.format(COLLECTION)

os.makedirs(FILES_DIR, exist_ok=True)
print('Building "{0}" -> {1}, audio in {2}/'.format(COLLECTION, DATAFRAME_FILENAME, FILES_DIR))

In [ ]:
def query_freesound(query, filter, sort, num_results):
    """Query Freesound and return the resulting sound objects."""
    pager = freesound_client.text_search(
        query=query,
        filter=filter,
        sort=sort,
        fields=','.join(FREESOUND_STORE_METADATA_FIELDS),
        group_by_pack=1,
        page_size=num_results,
    )
    pager.next_page()
    return list(pager)


def retrieve_sound_preview(sound, directory):
    """Download the high-quality OGG preview of a sound into `directory`."""
    return freesound.FSRequest.retrieve(
        sound.previews.preview_hq_ogg,
        freesound_client,
        os.path.join(directory, sound.previews.preview_hq_ogg.split('/')[-1]),
    )


def make_pandas_record(sound, directory):
    """The metadata stored for each sound, including where its preview landed."""
    record = {key: sound.as_dict()[key] for key in FREESOUND_STORE_METADATA_FIELDS}
    del record['previews']
    record['freesound_id'] = record.pop('id')
    record['path'] = os.path.join(directory, sound.previews.preview_hq_ogg.split('/')[-1])
    return record

In [ ]:
sounds = sum(
    [query_freesound(q['query'], q['filter'], q['sort'], q['num_results'])
     for q in COLLECTIONS[COLLECTION]],
    [],
)

for count, sound in enumerate(sounds):
    print('Downloading sound with id {0} [{1}/{2}]'.format(sound.id, count + 1, len(sounds)))
    retrieve_sound_preview(sound, FILES_DIR)

df = pd.DataFrame([make_pandas_record(s, FILES_DIR) for s in sounds])
df.to_csv(DATAFRAME_FILENAME)
print('Saved DataFrame with {0} entries! {1}'.format(len(df), DATAFRAME_FILENAME))

display(df)

## Attribution

Freesound sounds carry per-sound licenses. Anything CC-BY requires crediting the uploader,
and anything CC BY-NC cannot be used commercially. Write the credits out alongside the
collection so a published demo can be attributed to the sounds it was actually built from.

In [ ]:
print(df['license'].value_counts().to_string())

credits_filename = 'credits_{0}.txt'.format(COLLECTION)
with open(credits_filename, 'w') as f:
    for _, row in df.iterrows():
        f.write('{0} by {1} -- https://freesound.org/s/{2}/ -- {3}\n'.format(
            row['name'], row['username'], row['freesound_id'], row['license']))
print('\nWrote {0}'.format(credits_filename))